In [ ]:
# Create a new S3 bucket, programmatically via the AWS API, named dsan6000-sc2641
# Take the .parquet files you've stored locally in the "data" subfolder, and add them into a "subfolder" within this bucket called wikipedia-hourly.

In [6]:
# Imports
import io
import os
import uuid
import glob

import boto3
from boto3.s3.transfer import S3UploadFailedError
from botocore.exceptions import ClientError

In [13]:
def create_s3_bucket_and_upload_files(s3_resource, bucket_name, prefix, local_filepath):
    """

    Parameters
    ----------
    s3_resource:

    bucket_name: str
                 Name to be given to created S3 bucket
    prefix: str
            Prefix or subfolder for files to uploaded to in the S3 bucket
    local_filepath: str
                    Local path of files to upload to S3 bucket

    Returns
    -------
    None
    """

    #### Create S3 bucket
    # Initialize S3 bucket
    bucket = s3_resource.Bucket(bucket_name)

    # Identify Amazon region
    region = s3_resource.meta.client.meta.region_name

    # Create S3 bucket
    try:
        # Since us-east-1 doesn't accept a LocationConstraint, hardcoding it as below
        if region == "us-east-1":
            bucket.create()
        else:
            bucket.create(
                CreateBucketConfiguration={
                    "LocationConstraint": s3_resource.meta.client.meta.region_name
                }
            )
        print(f"S3 bucket {bucket_name} successfully created!")

    except ClientError as err:
        print(f"Tried and failed to create bucket {bucket_name}. Please review error:")
        print(f"\t{err.response['Error']['Code']}:{err.response['Error']['Message']}")


    #### Upload files to S3 bucket
    # Loop through files in given local file path
    for file_name in glob.glob(os.path.join(local_filepath, "*")):
        if not os.path.exists(file_name):
            print(f"Couldn't find file {file_name}. Are you sure it exists?")

        # Create S3 bucket object with prefix for the file
        key = os.path.join(prefix, os.path.basename(file_name))
        obj = bucket.Object(key)
        
        # Upload the file to the bucket
        try:
            obj.upload_file(file_name)
            print(
                f"Uploaded file {file_name} into bucket {bucket.name} with key {obj.key}."
            )
        except S3UploadFailedError as err:
            print(f"Couldn't upload file {file_name} to {bucket.name}.")
            print(f"\t{err}")

In [14]:
create_s3_bucket_and_upload_files(s3_resource = boto3.resource('s3'), bucket_name = 'dsan6000-sc2641', prefix = 'wikipedia-hourly', local_filepath = 'data/')

S3 bucket dsan6000-sc2641 successfully created!
Uploaded file data/20260901_090000_downloaded.parquet into bucket dsan6000-sc2641 with key wikipedia-hourly/20260901_090000_downloaded.parquet.
Uploaded file data/20260901_070000_downloaded.parquet into bucket dsan6000-sc2641 with key wikipedia-hourly/20260901_070000_downloaded.parquet.
Uploaded file data/20260901_150000_downloaded.parquet into bucket dsan6000-sc2641 with key wikipedia-hourly/20260901_150000_downloaded.parquet.
Uploaded file data/20260901_100000_downloaded.parquet into bucket dsan6000-sc2641 with key wikipedia-hourly/20260901_100000_downloaded.parquet.
Uploaded file data/20260901_050000_downloaded.parquet into bucket dsan6000-sc2641 with key wikipedia-hourly/20260901_050000_downloaded.parquet.
Uploaded file data/20260902_000000_downloaded.parquet into bucket dsan6000-sc2641 with key wikipedia-hourly/20260902_000000_downloaded.parquet.
Uploaded file data/20260901_110000_downloaded.parquet into bucket dsan6000-sc2641 with k